## MEMO – uso della 09_03

**Una volta sola:** pre-tagging, registrazione delle 15 predeterminate (09_02). Non rilanciare.

**Parametri:** `top_j` = chunk all'LLM, `top_k` = candidati del retrieval. Si cambiano nella cella 0 (o `bench.top_j = new`).
Fissare prima del `pre_edit`; non cambiare tra `pre_edit` e `post_edit`.
Dopo averli cambiati usa un `RUN_ID` nuovo (es. `post_edit_j7`) o cancella il vecchio `*_generation.jsonl`: `generate()` salta ciò che è già salvato.

**Ordine di esecuzione**
1. Cella 0 (init), una volta per kernel.
2. Pilota (opz.): 3–4 domande con `bench.generate("pilot", query_ids=[...])`; se C0 è già ~perfetto o molto basso, rivedi `top_j`.
3. Generazione con `RUN_ID = "pre_edit"` (cella 2). Riprendibile: se si interrompe, rilancia.
4. Backup sidecar (poz.): `shutil.copy(sidecar_path, sidecar_path.with_name("dst_benchmark_pre_edit.json"))`.
5. Editing: cella 1a (cambia `QID`, modifica nel widget), cella 1b per controllare. `bench.edit_status()` conta solo le modifiche query↔chunk delle predq.
6. Generazione con `RUN_ID = "post_edit"` (cella 2).
7. RAGAS (3.1 e 3.2) e report (4): lavorano sui file salvati, non sul sidecar. Imposta `RUN_ID` prima di ogni lancio.

**Altri esperimenti con modifiche diverse**
Cambiare `RUN_ID` rinomina ovviamente solo gli output. Necessario ripartire da una diversa istanziazione del sidecar (es la copia `dst_benchmark_pre_edit.json`); cambiare `sidecar_path` e un `RUN_ID` nuovo.

**Confronto prima/dopo**
Sulle righe C2 e C3 dei due run. C0 non dipende dalle modifiche: nel `post_edit` puoi valutare RAGAS solo su `configs=["C2_dist", "C3_dist_tags"]`.

**Attenzioni**
- `compare_retrieval_vs_reranking` (cella 1b) usa tutti i `top_k` chunk per il baseline, quindi solo per guardare, non per il confronto.
- NaN del giudice riportati e non sostituirli con 0.

In [ ]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)

QDIR = project_root / "data/queries/dst"
sidecar_path = project_root / "data/processed/dst/dst_benchmark.json"
queries_path = QDIR / "eval_queries.json"          # file unico (30 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/dst",
    top_j=7, top_k=10, top_n=7,       # stesso numero di chunk (top_j) per TUTTE le configurazioni
    # gold_path=QDIR / "gold_chunks.json",   # opzionale: {query_id: [chunk_id,...]} -> metriche retrieval senza LLM
)
print("!>> Handler e benchmark pronti")

In [ ]:
### PRE-TAGGING ###
candidate_tags = [
    "dense retrieval and retriever (DPR, Contriever, BM25, vector index)",
    "knowledge graph (entities, relations, triples, paths)",
    "graph communities and global sensemaking (community summaries, map-reduce)",
    "hypothetical document generation (HyDE)",
    "noise, distracting documents and context position",
    "self-reflection and critique (reflection tokens, critic, self-critique)",
    "chunking, context window and prompt length",
    "datasets and experimental setup"
]

#possibili altri tag
"hallucination and factual grounding"
"evaluation metrics and win rates (accuracy, nDCG, EM, LLM judge)"
"baselines and ablation study"
"prompting and reasoning (prompt template, mind map, chain of thought)"

batch_size = 100

handler.pretag_sidecar(
        sidecar_path=sidecar_path,
        candidate_tags=candidate_tags,
        batch_size=batch_size
    )

In [ ]:
### CONFIG PREDQ ### (se non gia' fatto in 09_02) + controllo similarita' per eventuali query in più sopra threshold
#bench.register_predetermined()
print(bench.check_similarity().head(5))

In [ ]:
### 1a ### EDITING INTERATTIVO - scegli la query: retrieval + reranking + widget (modifiche salvate nel sidecar)
import ipywidgets as widgets
from IPython.display import display

QID = "SPE_P03"                                   # cambia id per passare alla query successiva
q = bench.prepare(QID, order_by=["DIST", "TAGS"])
print(f"[{QID}] {q['query']}\n\nREFERENCE:\n{q['reference']}")

PHASE_MAP = {"1 Retrieval (Vettoriale)": "vectorial", "2 Retrieval (Distanze)": "distance",
             "Top-J Reranked": "reranked", "All Reranked": "all_reranked"}
phase_selector = widgets.ToggleButtons(options=list(PHASE_MAP), description="Fase Grafo:", button_style="info")
output_area = widgets.Output()

def on_phase_change(change):
    with output_area:
        output_area.clear_output(wait=True)
        handler.update_widget_view(PHASE_MAP[change["new"]])
        display(handler.widget)

phase_selector.observe(on_phase_change, names="value")
handler.update_widget_view("vectorial")
display(phase_selector, output_area)
with output_area:
    display(handler.widget)

In [ ]:
### 1b ### Dopo le modifiche: rilancia reranking e confronta i chunk (e la risposta, se vuoi) con la baseline
handler.run_top_j_distance_tag_reranking()
cmp = handler.compare_retrieval_vs_reranking()
print("baseline :", cmp["vectorial_baseline"]["chunk_ids"][:bench.top_j])
print("reranked :", cmp["post_reranking"]["chunk_ids"])
display(bench.edit_status().query("n_dist_edits > 0 or n_graph_tags > 0"))

In [ ]:
### 2 ### GENERAZIONE (Ollama) - riprendibile: se si interrompe basta rilanciare la cella
RUN_ID = "pre_edit"          # poi "post_edit" dopo i ritocchi manuali
gen = bench.generate(RUN_ID)
print(gen.groupby("config").seconds.mean().round(1))
print("predq non riconosciute/errate:", gen[~gen.predq_ok].query_id.unique())
# Metriche di retrieval senza giudice (solo se gold_path impostato)
print(bench.retrieval_summary(RUN_ID))

In [ ]:
### 3.1 ### VALUTAZIONE RAGAS con giudice esterno (piu' forte di llama3.2) - una configurazione alla volta, riprendibile
# Chiavi API da variabili d'ambiente (mai nel notebook). Esempi (verifica il nome del modello attuale):
from langchain_google_genai import ChatGoogleGenerativeAI      # pip install langchain-google-genai
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # GOOGLE_API_KEY
# from langchain_groq import ChatGroq; judge_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Prova prima su 1 configurazione per stimare token e tempo; max_workers basso per i rate limit del free tier
ragas_df = bench.evaluate_ragas(RUN_ID, judge_llm, max_workers=2, configs=["C0_baseline"])

In [ ]:
### 3.2 ### Poi tutte le configurazioni
ragas_df = bench.evaluate_ragas(RUN_ID, judge_llm, max_workers=2)
print(ragas_df.isna().sum())      # NaN = fallimenti del giudice: riportali nella tesi, non sostituirli con 0

In [ ]:
### 4 ### REPORT per la tesi
ragas_df = bench.load_ragas(RUN_ID)
display(bench.summary(ragas_df, by="category"))          # ambigue vs specifiche
display(bench.summary(ragas_df, by="is_predetermined"))  # in-sample (ritoccate) vs generalizzazione
display(bench.paired_vs_baseline(ragas_df))              # tutte le query
display(bench.paired_vs_baseline(ragas_df, subset=~ragas_df.is_predetermined))   # solo normali (generalizzazione)
ragas_df.to_csv(bench.out_dir / f"{RUN_ID}_ragas_all.csv", index=False)